## 기본 Chain 만들기 - AI 요리사

In [3]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


In [4]:
from langchain_openai import ChatOpenAI

# 기본 LLM 제공자: "groq" 또는 "openai" — get_llm() 호출 시 provider 를 생략하면 이 값을 사용
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from pprint import pprint

template_text = """
당신은 친절하고 전문적인 AI 요리사입니다.
사용자가 가진 재료: {ingredients}

위 재료를 활용해서 만들 수 있는 요리를 추천해 주세요.
출력 형식:
1. 추천 요리 이름
2. 필요한 재료 (추가로 있으면 좋은 양념 등 포함)
3. 단계별 조리법
"""

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

llm = get_llm(temperature=0.7, seed=42, provider="groq")  # 제공자를 직접 지정 (키/모델명은 공통 설정 셀의 get_llm 사용)

[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [6]:
from langchain_core.output_parsers import StrOutputParser

# 1. 사용자로부터 재료 입력받기
user_ingredients = input("가지고 있는 재료를 입력하세요 (예: 감자, 양파, 돼지고기): ")

# 2. 체인 구성 (프롬프트 | LLM | 출력파서)
chain = prompt_template | llm | StrOutputParser()

# 3. 입력받은 값을 딕셔너리 형태로 invoke에 전달
response = chain.invoke({"ingredients": user_ingredients})

# 4. 결과 출력
print(response)


**1. 추천 요리 이름**  
돼지고기 고추장 양념 구이 + 상추 쌈

---

**2. 필요한 재료**  

| 기본 재료 (이미 가지고 계신 것) | 추가로 준비하면 좋은 재료·양념 |
|------------------------------|------------------------------|
| 돼지고기 (목살·앞다리살 등 300 g) | 다진 마늘 1 큰술 |
| 상추 (쌈용) 8~10장 | 다진 파 1 큰술 |
| 고추장 2 큰술 | 설탕 1 큰술 (또는 물엿/꿀) |
| 양파 1/2개 | 참기름 1 작은술 |
|                              | 깨소금 약간 |
|                              | 식용유(구이용) 1 큰술 |
|                              | 소금·후추 약간 |
|                              | 레몬즙 또는 식초 1 작은술 (취향) |

> **Tip**: 돼지고기는 얇게 썰거나 1 cm 정도 두께로 썰면 양념이 잘 배어 맛있어요.

---

**3. 단계별 조리법**  

1. **재료 손질**  
   - 돼지고기는 한 입 크기로 썰어 주세요.  
   - 양파는 얇게 채 썰고, 상추는 깨끗이 씻어 물기를 제거해 둡니다.  

2. **양념장 만들기**  
   - 볼에 고추장 2 큰술, 다진 마늘 1 큰술, 다진 파 1 큰술, 설탕 1 큰술, 참기름 1 작은술, 깨소금 약간, 레몬즙(또는 식초) 1 작은술을 넣고 고루 섞어 줍니다.  

3. **고기 양념**  
   - 썰어 놓은 돼지고기에 소금·후추로 약간 밑간을 한 뒤, 만든 양념장을 넣고 10~15분 정도 골고루 버무려 주세요. (가능하면 냉장고에 30분 정도 재워 두면 더 깊은 맛).  

4. **양파 볶기**  
   - 팬에 식용유 1 큰술을 두르고 중불에서 양파 채를 넣어 투명해질 때까지 살짝 볶아 주세요. 볶은 양파는 따로 그릇에 담아 둡니다.  

5. **